In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

from scipy.fftpack import dct
from scipy.special import softmax

# ==========================================
# بارگذاری فایل صوتی
# ==========================================

audio_path = "sample.wav"

signal, sr = librosa.load(
    audio_path,
    sr=16000  # نمونه‌برداری با نرخ ۱۶ کیلوهرتز
)

# ==========================================
# پیش‌تأکید (Pre-Emphasis)
# y(n)=x(n)-a*x(n-1)
# ==========================================

a = 0.97  # ضریب پیش‌تأکید

pre_emphasis = np.append(
    signal[0],
    signal[1:] - a * signal[:-1]
)

# نمایش سیگنال اصلی و پیش‌تأکید شده
plt.figure(figsize=(12,4))
plt.plot(signal)
plt.title("Original Signal")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(pre_emphasis)
plt.title("Pre-Emphasis Signal")
plt.show()

# ==========================================
# تبدیل فوریه سریع (FFT)
# ==========================================

fft_original = np.abs(
    np.fft.rfft(signal)  # تبدیل فوریه برای سیگنال اصلی
)

fft_pre = np.abs(
    np.fft.rfft(pre_emphasis)  # تبدیل فوریه برای سیگنال پیش‌تأکید شده
)

freqs = np.fft.rfftfreq(
    len(signal),
    1/sr  # محاسبه فرکانس‌های متناظر
)

# نمایش طیف فرکانسی
plt.figure(figsize=(12,4))
plt.plot(freqs, fft_original)
plt.title("FFT Original")
plt.xlabel("Frequency")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(freqs, fft_pre)
plt.title("FFT After Pre-Emphasis")
plt.xlabel("Frequency")
plt.show()

# ==========================================
# پنجره همینگ (Hamming Window)
# ==========================================

frame_length = 512  # طول هر فریم

frame = signal[:frame_length]  # انتخاب فریم اول

window = np.hamming(frame_length)  # تولید پنجره همینگ

windowed = frame * window  # اعمال پنجره روی فریم

plt.figure(figsize=(10,4))
plt.plot(window)
plt.title("Hamming Window")
plt.show()

# ==========================================
# بانک فیلتر مل (Mel Filter Bank)
# ==========================================

mel_filter = librosa.filters.mel(
    sr=sr,
    n_fft=2048,  # تعداد نقاط FFT
    n_mels=40    # تعداد فیلترهای مل
)

plt.figure(figsize=(12,6))

for filt in mel_filter:
    plt.plot(filt)  # رسم هر فیلتر

plt.title("Mel Filter Bank")
plt.show()

# ==========================================
# طیف‌نگار مل (Mel Spectrogram)
# ==========================================

mel_spec = librosa.feature.melspectrogram(
    y=signal,
    sr=sr,
    n_mels=40
)

plt.figure(figsize=(10,5))

librosa.display.specshow(
    librosa.power_to_db(mel_spec),  # تبدیل به دسی‌بل
    x_axis='time',
    y_axis='mel'
)

plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()

# ==========================================
# تبدیل کسینوسی گسسته (DCT)
# ==========================================

log_mel = np.log(
    mel_spec + 1e-8  # لگاریتم برای جلوگیری از بی‌نهایت
)

mfcc_manual = dct(
    log_mel,
    axis=0,
    norm='ortho'  # نرمال‌سازی متعامد
)

plt.figure(figsize=(10,5))

librosa.display.specshow(
    mfcc_manual[:13],  # ۱۳ ضریب اول MFCC
    x_axis='time'
)

plt.colorbar()
plt.title("MFCC via DCT")
plt.show()

# ==========================================
# MFCC با کتابخانه Librosa
# ==========================================

mfcc_librosa = librosa.feature.mfcc(
    y=signal,
    sr=sr,
    n_mfcc=13
)

plt.figure(figsize=(10,5))

librosa.display.specshow(
    mfcc_librosa,
    x_axis='time'
)

plt.colorbar()
plt.title("MFCC Librosa")
plt.show()

# ==========================================
# تابع فعال‌سازی ReLU
# ==========================================

x = np.linspace(-10,10,1000)  # بازه ورودی

relu = np.maximum(
    0,
    x  # ReLU = max(0, x)
)

plt.figure(figsize=(8,4))
plt.plot(x,relu)
plt.title("ReLU")
plt.grid()
plt.show()

# ==========================================
# توابع Softmax و LogSoftmax
# ==========================================

z = np.array(
    [1,2,3,4,5]  # ورودی نمونه
)

soft = softmax(z)  # محاسبه Softmax

logsoft = np.log(soft)  # محاسبه LogSoftmax

# نمایش Softmax
plt.figure(figsize=(8,4))
plt.bar(range(len(soft)), soft)
plt.title("Softmax")
plt.show()

# نمایش LogSoftmax
plt.figure(figsize=(8,4))
plt.bar(range(len(logsoft)), logsoft)
plt.title("LogSoftmax")
plt.show()

print("Softmax")
print(soft)

print("\nLogSoftmax")
print(logsoft)

In [ ]:
import os
import glob
import librosa
import pandas as pd

DATASET_PATH = "data"  # مسیر دیتاست

results = []  # لیست نتایج

classes = sorted(os.listdir(DATASET_PATH))  # لیست کلاس‌ها

for class_name in classes:

    durations = []  # لیست مدت زمان‌ها

    files = glob.glob(
        os.path.join(
            DATASET_PATH,
            class_name,
            "*.wav"  # تمام فایل‌های WAV
        )
    )

    for file in files:

        y, sr = librosa.load(
            file,
            sr=None  # نرخ نمونه‌برداری اصلی
        )

        duration = len(y) / sr  # محاسبه مدت زمان

        durations.append(duration)

    results.append({
        "Class": class_name,
        "Min Duration": round(min(durations),2),  # حداقل مدت
        "Max Duration": round(max(durations),2),  # حداکثر مدت
        "Mean Duration": round(sum(durations)/len(durations),2)  # میانگین
    })

df = pd.DataFrame(results)  # تبدیل به دیتافریم

print(df)  # چاپ نتایج

df.to_csv(
    "duration_report.csv",
    index=False  # ذخیره بدون ایندکس
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.preprocessing import label_binarize

import numpy as np


def plot_confusion_matrix(
        y_true,
        y_pred,
        classes
):
    """رسم ماتریس درهم‌ریختگی"""

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=classes,
        yticklabels=classes
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()


def print_report(
        y_true,
        y_pred,
        classes
):
    """چاپ گزارش طبقه‌بندی"""

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=classes
        )
    )


def plot_roc(
        model,
        X_test,
        y_test,
        n_classes
):
    """رسم منحنی ROC برای هر کلاس"""

    y_score = model.predict_proba(
        X_test  # احتمال پیش‌بینی‌ها
    )

    y_bin = label_binarize(
        y_test,
        classes=np.arange(n_classes)  # دودویی‌سازی برچسب‌ها
    )

    plt.figure(figsize=(8,6))

    for i in range(n_classes):

        fpr, tpr, _ = roc_curve(
            y_bin[:,i],
            y_score[:,i]  # محاسبه نرخ مثبت کاذب و حقیقی
        )

        roc_auc = auc(
            fpr,
            tpr  # محاسبه مساحت زیر منحنی
        )

        plt.plot(
            fpr,
            tpr,
            label=f'Class {i} AUC={roc_auc:.2f}'
        )

    plt.plot(
        [0,1],
        [0,1],
        '--'  # خط تصادفی
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves")
    plt.legend()
    plt.show()

In [ ]:
import numpy as np

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.optimizers import SGD

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from scikeras.wrappers import KerasClassifier


# ==========================================
# سازنده مدل
# ==========================================

def create_model(
        optimizer='adam',
        dropout_rate=0.5
):
    """ایجاد مدل شبکه عصبی با معماری مشخص"""

    model = Sequential()

    # لایه ۱
    model.add(Dense(
        128,
        activation='relu',
        input_shape=(13,)  # ۱۳ ویژگی MFCC
    ))

    model.add(BatchNormalization())  # نرمال‌سازی دسته‌ای
    model.add(Dropout(dropout_rate))  # حذف تصادفی نورون‌ها

    # لایه ۲
    model.add(Dense(
        128,
        activation='relu'
    ))

    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # لایه ۳
    model.add(Dense(
        64,
        activation='relu'
    ))

    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # لایه ۴
    model.add(Dense(
        64,
        activation='relu'
    ))

    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # لایه ۵
    model.add(Dense(
        32,
        activation='relu'
    ))

    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # لایه خروجی
    model.add(Dense(
        5,  # ۵ کلاس
        activation='softmax'
    ))

    # انتخاب بهینه‌ساز
    if optimizer == 'adam':
        opt = Adam()
    elif optimizer == 'rmsprop':
        opt = RMSprop()
    else:
        opt = SGD()

    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# ==========================================
# جستجوی شبکه‌ای (Grid Search)
# ==========================================

def run_grid_search(X_train, y_train):
    """اجرای جستجوی شبکه‌ای برای یافتن بهترین هایپرپارامترها"""

    model = KerasClassifier(
        model=create_model,
        verbose=0
    )

    param_grid = {
        "optimizer": ["adam", "rmsprop", "sgd"],
        "batch_size": [16, 32, 64],
        "epochs": [50, 100],
        "dropout_rate": [0.3, 0.5]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=3,  # اعتبارسنجی متقابل ۳-تایی
        scoring='accuracy',
        n_jobs=-1  # استفاده از تمام پردازنده‌ها
    )

    grid.fit(
        X_train,
        y_train
    )

    print("\nBEST PARAMETERS")
    print(grid.best_params_)

    print("\nBEST SCORE")
    print(grid.best_score_)

    return grid.best_estimator_


# ==========================================
# اعتبارسنجی متقابل ۱۰-تایی
# ==========================================

def cross_validation(X, y):
    """اجرای اعتبارسنجی متقابل ۱۰-تایی"""

    model = KerasClassifier(
        model=create_model,
        optimizer='adam',
        dropout_rate=0.5,
        batch_size=32,
        epochs=100,
        verbose=0
    )

    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42  # برای تکرارپذیری
    )

    scores = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model.fit(
            X_train,
            y_train  # آموزش مدل
        )

        score = model.score(
            X_test,
            y_test  # ارزیابی مدل
        )

        scores.append(score)

    print("\n10-FOLD ACCURACY")
    print(np.mean(scores))

    return scores

In [ ]:
import matplotlib.pyplot as plt
import pickle

# بارگذاری تاریخچه آموزش
with open(
    "history.pkl",
    "rb"  # خواندن به صورت باینری
) as f:

    history = pickle.load(f)  # دسریالایز کردن

# منحنی دقت
plt.figure(figsize=(8,5))

plt.plot(
    history["accuracy"],
    label="Train"  # دقت آموزش
)

plt.plot(
    history["val_accuracy"],
    label="Validation"  # دقت اعتبارسنجی
)

plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# منحنی خطا
plt.figure(figsize=(8,5))

plt.plot(
    history["loss"],
    label="Train"  # خطای آموزش
)

plt.plot(
    history["val_loss"],
    label="Validation"  # خطای اعتبارسنجی
)

plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np

from sklearn.model_selection import train_test_split

from src.model_ann import run_grid_search
from src.model_ann import cross_validation

from src.evaluation import plot_confusion_matrix
from src.evaluation import print_report
from src.evaluation import plot_roc


# ---------------------------------
# بارگذاری ویژگی‌ها
# ---------------------------------

X = np.load("X.npy")  # ویژگی‌های MFCC
y = np.load("y.npy")  # برچسب‌ها

classes = [
    "Belly Pain",
    "Burping",
    "Discomfort",
    "Hungry",
    "Tired"  # ۵ کلاس گریه نوزاد
]

# ---------------------------------
# تقسیم داده‌ها
# ---------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.25,  # ۲۵٪ برای تست نهایی
    stratify=y,  # تقسیم طبقه‌بندی شده
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.60,  # ۶۰٪ از باقیمانده برای تست
    stratify=y_temp,
    random_state=42
)

# ---------------------------------
# جستجوی شبکه‌ای
# ---------------------------------

best_model = run_grid_search(
    X_train,
    y_train  # یافتن بهترین هایپرپارامترها
)

# ---------------------------------
# ارزیابی روی داده‌های تست
# ---------------------------------

y_pred = best_model.predict(
    X_test  # پیش‌بینی روی داده‌های تست
)

print_report(
    y_test,
    y_pred,
    classes  # گزارش طبقه‌بندی
)

plot_confusion_matrix(
    y_test,
    y_pred,
    classes  # ماتریس درهم‌ریختگی
)

plot_roc(
    best_model,
    X_test,
    y_test,
    len(classes)  # منحنی ROC
)

# ---------------------------------
# اعتبارسنجی متقابل ۱۰-تایی
# ---------------------------------

cross_validation(
    X,
    y  # ارزیابی نهایی مدل
)